In [1]:
from dataclasses import dataclass
from typing import Tuple

import numpy as np

import jax
import jax.numpy as jnp
import jax.random as jr
from flax import linen as nn
import scipy.io as sio
import numpy as np
import diffrax as dfx

from ckpt_io import load_model
from scipy.stats import wasserstein_distance

In [2]:
class VelocityMLP(nn.Module):
    """v_theta(t, z, x): concat [z, t, x] -> MLP -> R^d."""

    hidden_dims: Tuple[int, ...]
    dim_out: int

    @nn.compact
    def __call__(self, t, z, x):
        # t: () or (B,), z: (d,) or (B,d), x: (k,) or (B,k)
        # Broadcast t to match batch of z/x if needed
        if t.ndim == 0:
            t = t[None] if z.ndim == 1 else jnp.broadcast_to(t, (z.shape[0],))
        t = t[..., None]  # (B,1) or (1,1)

        # Ensure rank alignment
        if z.ndim == 1:
            z = z[None, :]  # (1,d)
        if x.ndim == 1:
            x = x[None, :]  # (1,k)

        h = jnp.concatenate([z, t, x], axis=-1)
        for width in self.hidden_dims:
            h = nn.silu(nn.Dense(width)(h))
        out = nn.Dense(self.dim_out)(h)  # (B, d)
        return out if out.shape[0] > 1 else out[0]  # return (d,) if single


# ============== Configs ==============
@dataclass
class SolverCfg:
    rtol: float = 1e-5
    atol: float = 1e-5
    max_steps: int = 1_000_000


def sample(
    key,
    params,
    x,
    z0=None,
    M=None,
):
    # Build once
    term = dfx.ODETerm(lambda t, y, args: velocity(params, model, t, y, args))

    def solve_single(z0i, x):
        sol = dfx.diffeqsolve(
            term,
            dfx.Tsit5(),
            t0=0.0,
            t1=1.0,
            dt0=None,
            y0=z0i,
            args=x,  # pass conditioning vector for this traj
            saveat=dfx.SaveAt(t1=True),
            stepsize_controller=dfx.PIDController(rtol=1e-5, atol=1e-5),
            max_steps=1_000_000,
        )
        return sol.ys

    # If you want n samples for the SAME x:
    # z0 = jr.normal(key, [x.shape[0], 4])
    if z0 is None:
        z0 = np.random.normal(size=[x.shape[0], 1])
        z0 = jnp.array(z0)
    else:
        # z0 = jnp.tile(z0, [M, 1])
        z0 = jnp.tile(z0[None, :, :], [M, 1, 1])
        z0 = jnp.transpose(z0, [1, 0, 2])
        z0 = z0.reshape([-1, z0.shape[-1]])
    x_fixed = x  # (k,)
    zT = jax.vmap(solve_single, in_axes=(0, 0))(z0, x_fixed)
    return zT, z0


# ============== Inference helpers (plug these into your sampler) ==============


def velocity(params, model, t, z, x):
    """Single-trajectory: t:(), z:(d,), x:(k,) -> (d,)"""
    return model.apply({"params": params}, t, z, x)

In [3]:
data = sio.loadmat("./data/data.mat")
x_data = data["x_data_test"].reshape([3, -1]).T
np.random.seed(88282)  # for reproducibility
M = 1_000
idx = np.random.choice(x_data.shape[0], x_data.shape[0], replace=False)[:M]
x_data = x_data[idx, :]

#### $\Delta x_1$

In [4]:
params, model, extras = load_model("./checkpoints/cfm_1", VelocityMLP)
x_mu = extras["x_mu"]
x_sd = extras["x_sd"]
y_mu = extras["y_mu"]
y_sd = extras["y_sd"]

In [5]:
N = 10_000
z0 = np.random.normal(size=[N, 1])
x = np.tile(x_data, [N, 1]).reshape([N * M, x_data.shape[1]])
key = jr.PRNGKey(72819)
z_samples, z0_samples = sample(
    key,
    params,
    x=(x - x_mu) / x_sd,
    z0=z0,
    M=M,
)
y_samples = z_samples[:, 0, :] * y_sd + y_mu

In [6]:
dt = 0.001
sigma = 0.1
y_samples = y_samples.reshape([N, M, 1])
x_samples = x.reshape([N, M, 3])
y_ref_samples = x_samples[..., 0:1] * x_samples[..., 2:3] * dt + sigma * np.sqrt(
    dt
) * np.random.normal(size=[N, M, 1])

In [7]:
w1 = []
for i in range(M):
    w1 += [wasserstein_distance(y_samples[:, i, 0], y_ref_samples[:, i, 0])]

np.mean(w1), np.std(w1)

(np.float64(0.00011496658311820177), np.float64(0.0001296951448324651))

#### $\Delta x_2$

In [8]:
params, model, extras = load_model("./checkpoints/cfm_2", VelocityMLP)
x_mu = extras["x_mu"]
x_sd = extras["x_sd"]
y_mu = extras["y_mu"]
y_sd = extras["y_sd"]

In [9]:
N = 10_000
z0 = np.random.normal(size=[N, 1])
x = np.tile(x_data, [N, 1]).reshape([N * M, x_data.shape[1]])
key = jr.PRNGKey(72819)
z_samples, z0_samples = sample(
    key,
    params,
    x=(x - x_mu) / x_sd,
    z0=z0,
    M=M,
)
y_samples = z_samples[:, 0, :] * y_sd + y_mu

In [10]:
dt = 0.001
sigma = 0.1
y_samples = y_samples.reshape([N, M, 1])
x_samples = x.reshape([N, M, 3])
y_ref_samples = -x_samples[..., 1:2] * x_samples[..., 2:3] * dt + sigma * np.sqrt(
    dt
) * np.random.normal(size=[N, M, 1])

In [11]:
w1 = []
for i in range(M):
    w1 += [wasserstein_distance(y_samples[:, i, 0], y_ref_samples[:, i, 0])]

np.mean(w1), np.std(w1)

(np.float64(0.00011593361353909948), np.float64(6.0287226731000064e-05))

#### $\Delta x_3$

In [12]:
params, model, extras = load_model("./checkpoints/cfm_3", VelocityMLP)
x_mu = extras["x_mu"]
x_sd = extras["x_sd"]
y_mu = extras["y_mu"]
y_sd = extras["y_sd"]

In [13]:
N = 10_000
z0 = np.random.normal(size=[N, 1])
x = np.tile(x_data, [N, 1]).reshape([N * M, x_data.shape[1]])
key = jr.PRNGKey(72819)
z_samples, z0_samples = sample(
    key,
    params,
    x=(x - x_mu) / x_sd,
    z0=z0,
    M=M,
)
y_samples = z_samples[:, 0, :] * y_sd + y_mu

In [14]:
dt = 0.001
sigma = 0.1
y_samples = y_samples.reshape([N, M, 1])
x_samples = x.reshape([N, M, 3])
y_ref_samples = (
    -x_samples[..., 0:1] ** 2 + x_samples[..., 1:2] ** 2
) * dt + sigma * np.sqrt(dt) * np.random.normal(size=[N, M, 1])

In [15]:
w1 = []
for i in range(M):
    w1 += [wasserstein_distance(y_samples[:, i, 0], y_ref_samples[:, i, 0])]


np.mean(w1), np.std(w1)

(np.float64(0.00015852272875701721), np.float64(0.00020579085454324614))